# this file show t-he used prompt to generate QA by bassing the context of the whole document (Word, PDF), task_code (F=GOV, EXE, DIS, EVL), subtaskname, and the number of Question that needed to be generated 

In [ ]:
def build_multimodal_message(context, task_code, subtask_name, image_list, nu_question):
    '''
    Role: Expert Requirements Engineer.
    Goal: Generate N unique, image-driven technical questions grounded in diagrams,
          where answers are primarily extracted from visual evidence and optionally
          supported by textual content.
    Strict output format required.
    '''

    task = tasks[task_code]
    task_name = task["name"]
    definition = task["definition"]
    subtask_definition = extract_subtask_definition(definition, subtask_name)
    #short_context = context[3000:]
    text_prompt = f"""
You are an expert Requirements Engineer specializing in diagram-centric analysis.

You are given:
• Document text
• One or more diagrams/images extracted from the document

Your task is to generate **image-driven requirements questions**.

====================================================
TASK CONTEXT
====================================================
TASK: {task_name}
SUBTASK: {subtask_name}

--- Subtask Definition & Guidance ---
{subtask_definition}
--- End of Definition ---

====================================================
CORE OBJECTIVE
====================================================
Generate **at least ONE - UP TO {nu_question} unique questions** that:
• Are **primarily derived from the diagrams/images**
• Require interpreting visual structure, flow, entities, or relationships
• May be supported by relevant textual content if needed
• Are explicitly aligned with the {subtask_name} subtask

====================================================
STRICT QUESTION RULES
====================================================
1. **Image-First Requirement**
   - Each question MUST rely on visual evidence from at least one diagram/image.
   - Textual content may only be used as *supporting* evidence.
   - Do NOT ask Compound Question.
   - Do NOT generate purely text-based questions.
   - **IGNORE TABLE OF CONTENTS:** Do not ask about section numbers, headers, or page listings found in the Index/ToC.
   - **IGNORE METADATA:** Do not ask about version numbers, authors, or copyright dates on the cover page.
   - **NO "PAGE 1" QUESTIONS:** Focus on technical content (Process flows, Architecture, Data models).

2. **Reasoning Depth**
   - Avoid trivial or cosmetic questions (e.g., colors, shapes, labels only).

3. **Acceptable Question Examples**
   - "Based on the sequence shown in the  process flow diagram, which component initiates the authentication request?"
   - "According to the architecture diagram, how does the Data Validation module interact with the API Gateway?"
   - "How many agents in Figure X ?"
   - "what type of interaction between Agent_A and Agent_B in Figure X?"
   - "explain the flow in the figure X that visualizes the initiation phase of the XYZ application?"
   - "How Actor/Agent X interact with XYZ Application and show which figure  explains that?"
   - "Explain the Use-case diagram for X function."
   - "Explain the diagram of Level-1 DFD for XYZ system in page N."

4. **Unacceptable Question Examples**
   - “What is written inside the box?”
   - “What color is the arrow?”

5. **Evidence Traceability**
   - Clearly identify:
     • Image name(s)
     • Supporting section(s)
     • Page number(s)

6. **EVIDENCE TYPE LABELING (MANDATORY)**
   - For each question, classify the evidence used as one of:
       • **Explicit** — the answer is directly visible in the diagram
         (labels, arrows, entity names, text inside shapes).
       • **Implicit** — the answer is inferred from visual logic or structure
         (ordering of steps, connectivity, hierarchy, data flow, interaction paths, or relationships across one or more diagrams).
   - The Evidence Type must be consistent with the Direct Answer explanation.

====================================================
DIFFICULTY ASSIGNMENT
====================================================
Assign a difficulty level for each question:

• **Easy (Single-hop)**
  - Single diagram
  - Direct visual interpretation
  - Minimal reasoning

• **Hard (Multi-hop)**
  - Multiple diagrams and/or pages, sections
  - Cross-diagram or diagram-text reasoning
  - Multi-step inference

====================================================
OUTPUT FORMAT (STRICT – NO DEVIATION)
====================================================
Use the following format EXACTLY for each question:

**Question X:** <Question text>
**Difficulty Level:** <Easy | Hard>
**Evidence Type:** <Explicit | Implicit>
**Direct Answer:** <Exact visual detail, label, or quoted description extracted from the diagram>
**Section Name:** <Relevant section(s)>
**Page Reference:** <Page number(s)>
**Generated Answer:** <Complete, well-reasoned answer grounded in visual and textual evidence>

====================================================
EXAMPLE
====================================================
**Question 1:** According to the authentication workflow diagram, which component initiates the login process?
**Difficulty Level:** Easy
**Evidence Type:** Explicit
**Direct Answer:** The “User Interface” component is the first element in the diagram and has an outgoing arrow to the Authentication Service.
**Section Name:** 4.1 Authentication Workflow
**Page Reference:** Page 22
**Generated Answer:** The login process is initiated by the User Interface, as shown by the first outgoing arrow leading to the Authentication Service.

**Question 2:** Based on the system architecture diagram, which component acts as the central mediator between the client and backend services?
**Difficulty Level:** Hard
**Evidence Type:** Implicit
**Direct Answer:** The API Gateway is connected to all backend services and serves as the only interface between the client and internal components.
**Section Name:** 5.2 System Architecture
**Page Reference:** Page 30
**Generated Answer:** Although not explicitly labeled as a mediator, the API Gateway’s central position and connectivity indicate that it coordinates communication between the client and backend services.

====================================================
DOCUMENT CONTEXT
====================================================
{context}
====================================================
DOCUMENT IMAGE
====================================================
{image_list}
"""

    user_content = [{"type": "text", "text": text_prompt}]

    if image_list:
        for img_path in image_list[:10]:
            base64_img = encode_image(img_path)
            if base64_img:
                user_content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{base64_img}"}
                })

    return user_content



## multimodal QA Generation prompt

In [ ]:
def build_text_message(context, task_code, subtask_name, nu_question):

    '''
    Role: Expert Requirements Engineer.
    Goal: Generate N unique, text-only requirements questions grounded strictly
          in the provided document text.
    Diagrams/images must NEVER be referenced.
    Strict output format required.
    '''

    task = tasks[task_code]
    task_name = task["name"]
    definition = task["definition"]
    subtask_definition = extract_subtask_definition(definition, subtask_name)


    text_prompt = f"""
You are an expert Requirements Engineer specializing in textual analysis of
Software Requirements Specifications.

====================================================
TASK CONTEXT:
====================================================
TASK: {task_name}
SUBTASK: {subtask_name}

--- Subtask Definition & Guidance ---
{subtask_definition}
--- End of Definition ---

====================================================
CORE OBJECTIVE
====================================================
Generate **exactly {nu_question} unique questions** that:
• Are derived **strictly and exclusively from the textual content**(ignore all images, diagrams, and other visuals).
• Are clearly aligned with the {subtask_name} subtask

====================================================
STRICT TEXT-ONLY RULES
====================================================
1. **NO DIAGRAMS OR IMAGES**
   - Strictly ignore all figures, diagrams, workflows, or images.
   - Do not reference, describe, or imply any visual content (e.g., "as shown", "illustrated", "depicted").

2. **TEXTUAL GROUNDING**
   - Each question MUST be answerable using:
       • a single page or section, OR
       • multiple pages and/or sections.
   - Explicitly identify the supporting Section(s) and Page(s).

3. **QUESTION QUALITY**
   - Only generate questions where the answer is 100% visible in the provided text.
   - Do NOT generate vague, generic, or overlapping questions.
   - Do NOT ask Compound Question.
   - Each question must target a the provided textual content.
   - NEVER leave any field empty.
   - NEVER assume outside technical details or industry standards not explicitly stated.
   - If the text does not contain enough detail for a subtask, generate fewer questions rather than hallucinating details.

4. **IMPLICIT TEXTUAL EVIDENCE IS ALLOWED**
   - The Direct Answer does NOT need to be a verbatim quote.
   - It MAY be inferred from:
       • section structure
       • numbering patterns
       • repeated or enumerated requirement entries.
   - When inference is used (e.g., counting requirements),
     the Direct Answer MUST explicitly explain how the conclusion
     was derived from the text.

5. **EVIDENCE TYPE LABELING (MANDATORY)**
   - For each question, classify the evidence used as one of:
       • Explicit — the answer is directly stated in the text.
       • Implicit — the answer is inferred from structure, counting,
         aggregation, or synthesis across sections.
   - The Evidence Type must be consistent with the Direct Answer explanation.
====================================================
DIFFICULTY ASSIGNMENT
====================================================
You must label difficulty based on the following retrieval patterns:

• **Easy (Single-hop)**:
  - The answer is found in one local span (one paragraph or section of text).
  - Requires simple lookup or minimal paraphrase.

• **Hard (Multi-hop)**:
  - The answer requires combining multiple pieces of evidence from DIFFERENT sections or pages of text.
  - Requires logical connection, aggregation (counting), or comparison across the document.

====================================================
OUTPUT FORMAT (STRICT – NO DEVIATION)
====================================================
Use the following format EXACTLY:

**Question X:** <Question text>
**Difficulty Level:** <Easy | Hard>
**Evidence Type:** <Explicit | Implicit>
**Direct Answer:** <Exact quote or faithful paraphrase from the document text>
**Section Name:** <Relevant section(s)>
**Page Reference:** <Page number(s)>
**Generated Answer:** <Complete, well-reasoned answer grounded in the cited text>


====================================================
DOCUMENT CONTEXT
====================================================
{context}
"""

    return text_prompt
